# SUMO Minimal Prototype / Scenario Validation

This notebook documents the SUMO scenario-validation layer for the Auckland public transport delay prediction project.

SUMO is used here as an added-value road-traffic scenario validation tool. It is not a live operations engine, not a full Auckland network simulation, and not proof of operational deployment success.

**Required interpretation:**

> SUMO outputs are scenario-estimated impacts and do not prove real-world operational success.


## Architecture Position

Project architecture:

```text
GTFS Static + GTFS-Realtime + Open-Meteo
-> AI Prediction
-> SHAP Explainability
-> Decision Engine
-> SUMO Scenario Validation
-> Streamlit Dashboard
```

SUMO supports the final operational-response validation step. It tests whether a Decision Engine-style intervention scenario produces improved scenario-level indicators compared with disruption conditions.


In [1]:
from pathlib import Path
import os
import shutil
import subprocess
import xml.etree.ElementTree as ET
import duckdb

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
SUMO_ROOT = PROJECT_ROOT / "sumo" / "minimal_prototype"
FALLBACK_DIR = SUMO_ROOT / "fallback_synthetic"
OSM_DIR = SUMO_ROOT / "osm_visual"
OSM_OUTPUT_DIR = OSM_DIR / "outputs"

PROJECT_ROOT, SUMO_ROOT, OSM_DIR

(WindowsPath('d:/Yoobee/Term 3/Capstone/ai-dss-auckland-public-transport'),
 WindowsPath('d:/Yoobee/Term 3/Capstone/ai-dss-auckland-public-transport/sumo/minimal_prototype'),
 WindowsPath('d:/Yoobee/Term 3/Capstone/ai-dss-auckland-public-transport/sumo/minimal_prototype/osm_visual'))

## 1. Local SUMO Availability Check

This project can display saved SUMO outputs without running SUMO live. For this module, SUMO was also installed locally so that the scenarios can be run and inspected in SUMO GUI.


In [17]:
sumo_home = os.environ.get("SUMO_HOME")
sumo_path = shutil.which("sumo")
sumo_gui_path = shutil.which("sumo-gui")
netconvert_path = shutil.which("netconvert")

print("SUMO_HOME:", sumo_home)
print("sumo:", sumo_path)
print("sumo-gui:", sumo_gui_path)
print("netconvert:", netconvert_path)

if sumo_path:
    result = subprocess.run([sumo_path, "--version"], capture_output=True, text=True)
    print(result.stdout.splitlines()[0] if result.stdout else result.stderr.splitlines()[0])

SUMO_HOME: C:\Program Files (x86)\Eclipse\Sumo\
sumo: C:\Program Files (x86)\Eclipse\Sumo\bin\sumo.EXE
sumo-gui: C:\Program Files (x86)\Eclipse\Sumo\bin\sumo-gui.EXE
netconvert: C:\Program Files (x86)\Eclipse\Sumo\bin\netconvert.EXE
Eclipse SUMO sumo 1.26.0


## 2. Route Selection Evidence

The route-selection rule is to use a top delayed non-special route, excluding routes that start with `S`. Because SUMO is a road-traffic simulator, the selected route must also be road-based. GTFS Static `route_type` is used to identify the transport mode.

GTFS route type mapping used here:

```text
2 = Train/Rail
3 = Bus
4 = Ferry
```


In [32]:
model_baseline_parquet_path = DATA_PROCESSED / "parquet" / "decision_engine_model_baseline.parquet"
duckdb_path = DATA_PROCESSED / "duckdb" / "gtfs_realtime.duckdb"
routes_path = PROJECT_ROOT / "data" / "raw" / "gtfs_static" / "routes.txt"

routes = pd.read_csv(routes_path)
con = duckdb.connect(str(duckdb_path), read_only=True)
final_dataset_source = "DuckDB view: decision_engine_model_baseline"

top_bus_routes = con.execute("""
with route_stats as (
  select
    route_id,
    any_value(service_type) as service_type,
    any_value(route_display_name) as route_display_name,
    any_value(route_corridor_name) as route_corridor_name,
    any_value(route_type) as route_type,
    count(*) as records,
    count(distinct trip_id) as unique_trips,
    round(avg(delay_minutes), 2) as avg_delay_min,
    round(avg(abs(delay_minutes)), 2) as avg_abs_delay_min,
    round(max(delay_minutes), 2) as max_delay_min,
    sum(case when delay_risk = 'Severe' then 1 else 0 end) as severe_records,
    sum(case when delay_risk = 'High' then 1 else 0 end) as high_records,
    sum(case when recommended_action = 'Deploy standby bus or supervisor review' then 1 else 0 end) as standby_records,
    min(collection_date) as first_date,
    max(collection_date) as last_date
  from decision_engine_model_baseline
  where coalesce(is_special_route, false) = false
    and route_type = 3
    and not starts_with(route_id, 'S')
  group by route_id
)
select *
from route_stats
where records >= 100
order by avg_delay_min desc, severe_records desc, records desc
limit 20
""").fetchdf()

top_bus_routes


,route_id,service_type,route_display_name,route_corridor_name,route_type,records,unique_trips,avg_delay_min,avg_abs_delay_min,max_delay_min,severe_records,high_records,standby_records,first_date,last_date
0,128-202,Bus,128,128 - Hibiscus Coast Station / Helensville,3,5958,42,8.55,11.44,40.57,1429.0,239.0,1429.0,2026-05-09,2026-06-04
1,126-203,Bus,126,126 - Albany / Westgate Via Riverhead,3,6635,107,3.86,5.29,111.77,64.0,116.0,64.0,2026-05-09,2026-06-04
2,807-202,Bus,807,807 - Devonport Wharf / Cheltenham Loop,3,6115,56,3.55,3.65,61.38,10.0,107.0,10.0,2026-05-09,2026-06-04
3,376-203,Bus,376,376 - Papakura Shops / Auranga Dr Via Drury,3,8882,192,2.92,4.01,61.68,64.0,151.0,64.0,2026-05-09,2026-06-04
4,501-217,Bus,501,501 - Kennedy Point / Matiatia Wharf,3,6700,59,2.91,3.03,21.60,0.0,32.0,0.0,2026-05-09,2026-06-04
5,392-203,Bus,392,392 - Pukekohe Station,3,9065,122,2.57,2.95,16.45,0.0,13.0,0.0,2026-05-09,2026-06-04
6,981-221,Bus,981,981 - Waiwera / Hibiscus Coast Station,3,15535,160,2.56,3.33,59.18,29.0,249.0,29.0,2026-05-09,2026-06-04
7,RBN-402,Bus,RBN,RBN - Westfield Newmarket / Britomart Rail Bus,3,224,34,2.43,2.74,8.40,0.0,0.0,0.0,2026-05-15,2026-05-16
8,856-203,Bus,856,856 - Albany Station / Browns Bay,3,24675,303,2.19,3.72,38.93,55.0,344.0,55.0,2026-05-09,2026-06-04
9,83-203,Bus,83,83 - Massey University / Takapuna Via Browns Bay,3,34930,483,2.19,3.50,48.35,31.0,234.0,31.0,2026-05-09,2026-06-04


The top delayed non-S route was `HUIA-404`, but GTFS Static shows it is `route_type = 2`, meaning rail/train. Because this SUMO prototype is road-traffic validation only, `HUIA-404` is excluded from the live SUMO road scenario.

The next suitable delayed non-S road-based candidate is `128-202`, where `route_type = 3`, meaning bus.


In [46]:
selected_route = "128-202"
evidence_period_start = "2026-05-09"
evidence_period_end = "2026-06-04"

route_context = top_bus_routes[top_bus_routes["route_id"] == selected_route].iloc[0].to_dict()
pd.DataFrame([route_context])


,route_id,service_type,route_display_name,route_corridor_name,route_type,records,unique_trips,avg_delay_min,avg_abs_delay_min,max_delay_min,severe_records,high_records,standby_records,first_date,last_date
0,128-202,Bus,128,128 - Hibiscus Coast Station / Helensville,3,5958,42,8.55,11.44,40.57,1429.0,239.0,1429.0,2026-05-09,2026-06-04


In [59]:
route_risk_action_counts = con.execute("""
select
  delay_risk,
  recommended_action,
  count(*) as records,
  round(count(*) * 100.0 / sum(count(*)) over (), 2) as share_pct
from decision_engine_model_baseline
where route_id = ?
group by delay_risk, recommended_action
order by case delay_risk
  when 'Low' then 1
  when 'Medium' then 2
  when 'High' then 3
  when 'Severe' then 4
  else 5
end
""", [selected_route]).fetchdf()

route_risk_action_counts


,delay_risk,recommended_action,records,share_pct
0,Low,No operational action required,3296,55.32
1,Medium,Monitor route conditions,994,16.68
2,High,Adjust service headway,239,4.01
3,Severe,Deploy standby bus or supervisor review,1429,23.98


### Final Model-Baseline Evidence Used for Scenario Assumptions

The OSM scenarios are informed by the final model-baseline dataset for Route `128-202` across `2026-05-09` to `2026-06-04`. This replaces the earlier single-day checkpoint evidence and keeps the SUMO module aligned with the final project dataset.

The aggregate evidence provides the delay and Decision Engine context used to describe the disruption and intervention scenarios. The SUMO scenario remains evidence-informed rather than calibrated to real Auckland traffic volumes.


In [71]:
route_final_evidence = con.execute("""
select
  route_id as selected_route,
  any_value(service_type) as service_type,
  any_value(route_display_name) as route_display_name,
  any_value(route_corridor_name) as route_corridor_name,
  min(collection_date) as evidence_period_start,
  max(collection_date) as evidence_period_end,
  count(*) as records,
  count(distinct trip_id) as unique_trips,
  round(avg(delay_minutes), 2) as avg_delay_min,
  round(avg(abs(delay_minutes)), 2) as avg_absolute_delay_min,
  round(max(delay_minutes), 2) as max_delay_min,
  sum(case when delay_risk = 'Severe' then 1 else 0 end) as severe_records,
  sum(case when delay_risk = 'High' then 1 else 0 end) as high_records,
  sum(case when recommended_action = 'Deploy standby bus or supervisor review' then 1 else 0 end) as standby_records
from decision_engine_model_baseline
where route_id = ?
group by route_id
""", [selected_route]).fetchdf().iloc[0].to_dict()

route_final_evidence


{'selected_route': '128-202',
 'service_type': 'Bus',
 'route_display_name': '128',
 'route_corridor_name': '128 - Hibiscus Coast Station / Helensville',
 'evidence_period_start': '2026-05-09',
 'evidence_period_end': '2026-06-04',
 'records': 5958,
 'unique_trips': 42,
 'avg_delay_min': 8.55,
 'avg_absolute_delay_min': 11.44,
 'max_delay_min': 40.57,
 'severe_records': 1429.0,
 'high_records': 239.0,
 'standby_records': 1429.0}

In [82]:
risk_counts = con.execute("""
select delay_risk, count(*) as records
from decision_engine_model_baseline
where route_id = ?
group by delay_risk
order by case delay_risk
  when 'Low' then 1
  when 'Medium' then 2
  when 'High' then 3
  when 'Severe' then 4
  else 5
end
""", [selected_route]).fetchdf()

recommendation_counts = con.execute("""
select recommended_action, count(*) as records
from decision_engine_model_baseline
where route_id = ?
group by recommended_action
order by records desc
""", [selected_route]).fetchdf()

display(risk_counts)
display(recommendation_counts)


,delay_risk,records
0,Low,3296
1,Medium,994
2,High,239
3,Severe,1429


,recommended_action,records
0,No operational action required,3296
1,Deploy standby bus or supervisor review,1429
2,Monitor route conditions,994
3,Adjust service headway,239


## 3. Route Context From GTFS Static

GTFS Static is used to provide operator-friendly context for the dashboard. For Route `128-202`, representative GTFS trips show the service between Hibiscus Coast Station and the Helensville / Kaipara Medical Centre area.


In [92]:
trips_path = PROJECT_ROOT / "data" / "raw" / "gtfs_static" / "trips.txt"
stop_times_path = PROJECT_ROOT / "data" / "raw" / "gtfs_static" / "stop_times.txt"
stops_path = PROJECT_ROOT / "data" / "raw" / "gtfs_static" / "stops.txt"

trips = pd.read_csv(trips_path)
route_trips = trips[trips["route_id"] == selected_route]
representative_trips = route_trips.groupby("direction_id", as_index=False).first()[["direction_id", "trip_id", "trip_headsign", "shape_id"]]
representative_trips

,direction_id,trip_id,trip_headsign,shape_id
0,0,1339-12801-21900-2-2ca0203b,Hibiscus Coast Station To Helensville,1339-12801-3dc989f9
1,1,1339-12802-19200-2-62093b12,Helensville To Hibiscus Coast Station,1339-12802-7fceb8dd


In [101]:
stop_times = pd.read_csv(stop_times_path)
stops = pd.read_csv(stops_path)

endpoint_rows = []
for _, trip in representative_trips.iterrows():
    trip_stop_times = stop_times[stop_times["trip_id"] == trip["trip_id"]].sort_values("stop_sequence")
    for role, row in [("start", trip_stop_times.iloc[0]), ("end", trip_stop_times.iloc[-1])]:
        stop = stops[stops["stop_id"] == row["stop_id"]].iloc[0]
        endpoint_rows.append({
            "direction_id": trip["direction_id"],
            "trip_headsign": trip["trip_headsign"],
            "role": role,
            "stop_id": row["stop_id"],
            "stop_name": stop["stop_name"],
            "stop_lat": stop["stop_lat"],
            "stop_lon": stop["stop_lon"],
        })

route_endpoints = pd.DataFrame(endpoint_rows)
route_endpoints

,direction_id,trip_headsign,role,stop_id,stop_name,stop_lat,stop_lon
0,0,Hibiscus Coast Station To Helensville,start,4982-1f78709d,Stop B Hibiscus Coast,-36.62406,174.66652
1,0,Hibiscus Coast Station To Helensville,end,4923-31817098,Kaipara Medical Centre,-36.67590,174.45083
2,1,Helensville To Hibiscus Coast Station,start,4923-31817098,Kaipara Medical Centre,-36.67590,174.45083
3,1,Helensville To Hibiscus Coast Station,end,4982-1f78709d,Stop B Hibiscus Coast,-36.62406,174.66652


## 4. SUMO Prototype File Structure

Two SUMO folders are maintained:

- `fallback_synthetic`: a simple synthetic corridor used as a controlled fallback.
- `osm_visual`: an OSM-based Helensville road network used for better visual context around the Route 128 endpoint.

Both use the same scenario logic: baseline, disruption, and intervention.


## 5. SUMO File Reproducibility Notes

This project does not provide a full SUMO tutorial. SUMO is used only as a scenario-validation layer, and users who wish to regenerate the network or scenario files are expected to understand basic SUMO workflows.

The SUMO files were created in two stages.

First, a fallback synthetic corridor was created. This was used to confirm that SUMO was installed correctly, that the project could represent baseline, disruption, and intervention conditions, and that SUMO XML outputs could later be summarized into lightweight dashboard CSVs. The fallback scenario is intentionally simple and is not intended to represent Auckland.

Second, an OpenStreetMap-based visual scenario was created for Route `128-202`. The OSM scenario was introduced because it gives better visual context for the Helensville / Kaipara Medical Centre endpoint of the selected bus route. It uses the same scenario logic as the fallback version, but the road network is based on a small OSM extract rather than a synthetic corridor.

The OSM extract was converted into a SUMO network using `netconvert` with left-hand traffic enabled:

```powershell
netconvert --osm-files helensville_route128_visual.osm --lefthand --output-file helensville_route128_visual.net.xml
```

The resulting OSM visual scenario files are stored in:

```text
sumo/minimal_prototype/osm_visual/
```

The finalized scenario outputs were generated using command-line SUMO:

```powershell
sumo -c osm_baseline.sumocfg
sumo -c osm_disruption.sumocfg
sumo -c osm_intervention.sumocfg
```

These commands generate SUMO XML outputs in:

```text
sumo/minimal_prototype/osm_visual/outputs/
```

The notebook then parses the generated XML files and summarizes them into lightweight CSV outputs for the dashboard.

The `.rou.xml` files are evidence-informed scenario assumptions. They are linked to final model-baseline Route `128-202` evidence from `2026-05-09` to `2026-06-04` and the Decision Engine recommendation, but they are not calibrated real-world traffic models.

SUMO outputs are scenario-estimated impacts and do not prove real-world operational success.


In [109]:
def list_files(path):
    return pd.DataFrame(
        [{"file": p.name, "size_bytes": p.stat().st_size} for p in sorted(path.iterdir())]
    )

print("Fallback synthetic files")
display(list_files(FALLBACK_DIR))

print("OSM visual files")
display(list_files(OSM_DIR))

Fallback synthetic files


,file,size_bytes
0,baseline.rou.xml,1150
1,baseline.sumocfg,799
2,disruption.rou.xml,1157
3,disruption.sumocfg,805
4,intervention.rou.xml,1167
5,intervention.sumocfg,811
6,outputs,4096
7,route_128_corridor.edg.xml,1411
8,route_128_corridor.net.xml,7851
9,route_128_corridor.nod.xml,477


OSM visual files


,file,size_bytes
0,helensville_route128_visual.net.xml,291009
1,helensville_route128_visual.osm,1394511
2,osm_baseline.rou.xml,1648
3,osm_baseline.sumocfg,823
4,osm_disruption.rou.xml,1898
5,osm_disruption.sumocfg,829
6,osm_intervention.rou.xml,1871
7,osm_intervention.sumocfg,835
8,osm_route128_stops.add.xml,1115
9,outputs,4096


## 5. Manual SUMO Run Instructions

Run these commands manually from PowerShell to regenerate SUMO XML outputs:

```powershell
cd "D:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\sumo\minimal_prototype\osm_visual"

sumo -c osm_baseline.sumocfg
sumo -c osm_disruption.sumocfg
sumo -c osm_intervention.sumocfg
```

For visual inspection in SUMO GUI:

```powershell
sumo-gui -c osm_baseline.sumocfg
sumo-gui -c osm_disruption.sumocfg
sumo-gui -c osm_intervention.sumocfg
```

Use the same configured duration for each OSM scenario: `1800` seconds.


## 6. Read Actual SUMO Outputs

The following cells parse the actual XML output files generated by SUMO. These are not manually fabricated values.

Scenario assumptions are designed by us; simulation outputs are generated by SUMO.


In [116]:
tripinfo_files = {
    "Baseline Operations": OSM_OUTPUT_DIR / "osm_baseline_tripinfo.xml",
    "Disruption Scenario": OSM_OUTPUT_DIR / "osm_disruption_tripinfo.xml",
    "Intervention Scenario": OSM_OUTPUT_DIR / "osm_intervention_tripinfo.xml",
}

summary_files = {
    "Baseline Operations": OSM_OUTPUT_DIR / "osm_baseline_summary.xml",
    "Disruption Scenario": OSM_OUTPUT_DIR / "osm_disruption_summary.xml",
    "Intervention Scenario": OSM_OUTPUT_DIR / "osm_intervention_summary.xml",
}

for scenario, path in tripinfo_files.items():
    print(scenario, path.exists(), path)

Baseline Operations True d:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\sumo\minimal_prototype\osm_visual\outputs\osm_baseline_tripinfo.xml
Disruption Scenario True d:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\sumo\minimal_prototype\osm_visual\outputs\osm_disruption_tripinfo.xml
Intervention Scenario True d:\Yoobee\Term 3\Capstone\ai-dss-auckland-public-transport\sumo\minimal_prototype\osm_visual\outputs\osm_intervention_tripinfo.xml


In [122]:
def parse_tripinfo(path):
    tree = ET.parse(path)
    rows = []
    for trip in tree.getroot().iter("tripinfo"):
        rows.append({
            "vehicle_id": trip.attrib.get("id"),
            "vehicle_type": trip.attrib.get("vType"),
            "duration_sec": float(trip.attrib.get("duration", 0)),
            "time_loss_sec": float(trip.attrib.get("timeLoss", 0)),
            "waiting_time_sec": float(trip.attrib.get("waitingTime", 0)),
            "route_length_m": float(trip.attrib.get("routeLength", 0)),
        })
    return pd.DataFrame(rows)

trip_metrics = []
trip_details = {}

for scenario, path in tripinfo_files.items():
    df = parse_tripinfo(path)
    trip_details[scenario] = df
    trip_metrics.append({
        "scenario_name": scenario,
        "completed_trips": len(df),
        "avg_duration_sec": round(df["duration_sec"].mean(), 2) if len(df) else 0,
        "max_duration_sec": round(df["duration_sec"].max(), 2) if len(df) else 0,
        "avg_time_loss_sec": round(df["time_loss_sec"].mean(), 2) if len(df) else 0,
        "max_time_loss_sec": round(df["time_loss_sec"].max(), 2) if len(df) else 0,
        "avg_waiting_time_sec": round(df["waiting_time_sec"].mean(), 2) if len(df) else 0,
    })

trip_metrics_df = pd.DataFrame(trip_metrics)
trip_metrics_df

,scenario_name,completed_trips,avg_duration_sec,max_duration_sec,avg_time_loss_sec,max_time_loss_sec,avg_waiting_time_sec
0,Baseline Operations,78,35.49,42.0,4.92,8.84,0.0
1,Disruption Scenario,238,67.04,71.0,9.91,12.05,0.0
2,Intervention Scenario,123,41.89,46.0,5.39,8.07,0.0


In [127]:
def parse_last_summary_step(path):
    tree = ET.parse(path)
    steps = list(tree.getroot().iter("step"))
    if not steps:
        return {}
    last = steps[-1].attrib
    return {
        "summary_steps": len(steps),
        "last_time_sec": float(last.get("time", 0)),
        "loaded": int(last.get("loaded", 0)),
        "inserted": int(last.get("inserted", 0)),
        "ended": int(last.get("ended", 0)),
        "arrived": int(last.get("arrived", 0)),
        "halting": int(last.get("halting", 0)),
        "mean_travel_time_sec": float(last.get("meanTravelTime", -1)),
        "mean_speed": float(last.get("meanSpeed", -1)),
    }

summary_metrics = []
for scenario, path in summary_files.items():
    row = {"scenario_name": scenario}
    row.update(parse_last_summary_step(path))
    summary_metrics.append(row)

summary_metrics_df = pd.DataFrame(summary_metrics)
summary_metrics_df

,scenario_name,summary_steps,last_time_sec,loaded,inserted,ended,arrived,halting,mean_travel_time_sec,mean_speed
0,Baseline Operations,1800,1799.0,78,78,78,78,0,35.49,-1.00
1,Disruption Scenario,1800,1799.0,247,247,238,238,0,67.04,5.91
2,Intervention Scenario,1800,1799.0,125,125,123,123,0,41.89,10.28


## 7. Scenario Comparison

Because completed trip counts differ across scenarios, results should be presented as scenario-level impacts rather than a like-for-like traffic-volume experiment.

The disruption scenario intentionally has heavier traffic pressure. The intervention scenario represents a managed response condition with reduced pressure and adjusted bus headway assumptions.


In [131]:
comparison = trip_metrics_df.merge(summary_metrics_df, on="scenario_name", how="left")
comparison

,scenario_name,completed_trips,avg_duration_sec,max_duration_sec,avg_time_loss_sec,max_time_loss_sec,avg_waiting_time_sec,summary_steps,last_time_sec,loaded,inserted,ended,arrived,halting,mean_travel_time_sec,mean_speed
0,Baseline Operations,78,35.49,42.0,4.92,8.84,0.0,1800,1799.0,78,78,78,78,0,35.49,-1.00
1,Disruption Scenario,238,67.04,71.0,9.91,12.05,0.0,1800,1799.0,247,247,238,238,0,67.04,5.91
2,Intervention Scenario,123,41.89,46.0,5.39,8.07,0.0,1800,1799.0,125,125,123,123,0,41.89,10.28


In [134]:
disruption_loss = comparison.loc[comparison["scenario_name"] == "Disruption Scenario", "avg_time_loss_sec"].iloc[0]
intervention_loss = comparison.loc[comparison["scenario_name"] == "Intervention Scenario", "avg_time_loss_sec"].iloc[0]

improvement_pct = ((disruption_loss - intervention_loss) / disruption_loss * 100) if disruption_loss else 0
print(f"Estimated intervention improvement vs disruption based on average SUMO time-loss: {improvement_pct:.1f}%")

Estimated intervention improvement vs disruption based on average SUMO time-loss: 45.6%


## 8. Dashboard-Ready CSV Generation

The following cell writes the lightweight CSV files used by the dashboard/presentation layer.

Only run this after confirming the actual SUMO outputs above are acceptable.


In [ ]:
WRITE_OUTPUTS = False

if WRITE_OUTPUTS:
    baseline = comparison[comparison["scenario_name"] == "Baseline Operations"].iloc[0]
    disruption = comparison[comparison["scenario_name"] == "Disruption Scenario"].iloc[0]
    intervention = comparison[comparison["scenario_name"] == "Intervention Scenario"].iloc[0]

    def min_from_seconds(value):
        return round(float(value) / 60, 2)

    baseline_duration = float(baseline["avg_duration_sec"])
    def congestion_index(row):
        return round(float(row["avg_duration_sec"]) / baseline_duration * 35, 2) if baseline_duration else 0

    service_type = route_final_evidence["service_type"]
    route_display_name = route_final_evidence["route_display_name"]
    route_corridor_name = route_final_evidence["route_corridor_name"]
    evidence_period = f"{route_final_evidence['evidence_period_start']} to {route_final_evidence['evidence_period_end']}"

    common_route_fields = {
        "selected_route": selected_route,
        "service_type": service_type,
        "route_display_name": route_display_name,
        "route_corridor_name": route_corridor_name,
    }

    sumo_scenarios = pd.DataFrame([
        {
            "scenario_name": "Baseline Operations",
            "description": f"OSM-based Helensville visual scenario for Route 128 bus context under normal traffic pressure; {int(baseline['completed_trips'])} completed trips in finalized command-line SUMO run. Baseline is a comparison condition, while final dataset evidence covers {evidence_period}.",
            **common_route_fields,
            "avg_delay_min": min_from_seconds(baseline["avg_time_loss_sec"]),
            "traffic_congestion_index": congestion_index(baseline),
            "recommended_action": "No operational action required",
        },
        {
            "scenario_name": "Disruption Scenario",
            "description": f"OSM-based Helensville visual scenario for Route 128 bus context with intentionally heavier traffic pressure informed by final model-baseline evidence for {evidence_period}: {int(route_final_evidence['records'])} records, {int(route_final_evidence['unique_trips'])} unique trips, avg delay {route_final_evidence['avg_delay_min']} min, max delay {route_final_evidence['max_delay_min']} min, and {int(route_final_evidence['severe_records'])} severe records; {int(disruption['completed_trips'])} completed trips in finalized command-line SUMO run.",
            **common_route_fields,
            "avg_delay_min": min_from_seconds(disruption["avg_time_loss_sec"]),
            "traffic_congestion_index": congestion_index(disruption),
            "recommended_action": "Deploy standby bus or supervisor review",
        },
        {
            "scenario_name": "Intervention Scenario",
            "description": f"OSM-based Helensville visual scenario for Route 128 bus context with reduced road pressure, adjusted service headway, and supervisor-managed response based on the Decision Engine recommendation; {int(intervention['completed_trips'])} completed trips in finalized command-line SUMO run.",
            **common_route_fields,
            "avg_delay_min": min_from_seconds(intervention["avg_time_loss_sec"]),
            "traffic_congestion_index": congestion_index(intervention),
            "recommended_action": "Adjust service headway / deploy standby bus or supervisor review",
        },
    ])

    sumo_validation_results = pd.DataFrame([
        {
            "scenario_name": "Baseline Operations",
            **common_route_fields,
            "avg_delay_min": min_from_seconds(baseline["avg_time_loss_sec"]),
            "max_delay_min": min_from_seconds(baseline["max_time_loss_sec"]),
            "congestion_index": congestion_index(baseline),
            "queue_impact": "Low",
            "service_reliability": "Stable",
            "validation_interpretation": "Baseline OSM visual scenario represents normal road-traffic pressure for the Route 128 bus endpoint context near Kaipara Medical Centre / Helensville. It provides the reference condition for comparing disruption and intervention behaviour. Output was generated from finalized command-line SUMO XML.",
        },
        {
            "scenario_name": "Disruption Scenario",
            **common_route_fields,
            "avg_delay_min": min_from_seconds(disruption["avg_time_loss_sec"]),
            "max_delay_min": min_from_seconds(disruption["max_time_loss_sec"]),
            "congestion_index": congestion_index(disruption),
            "queue_impact": "High",
            "service_reliability": "Degraded",
            "validation_interpretation": f"Disruption OSM visual scenario intentionally increases traffic pressure using final model-baseline Route 128 evidence from {evidence_period} as scenario context. It produced higher average duration and time-loss indicators than baseline. Output was generated from finalized command-line SUMO XML.",
        },
        {
            "scenario_name": "Intervention Scenario",
            **common_route_fields,
            "avg_delay_min": min_from_seconds(intervention["avg_time_loss_sec"]),
            "max_delay_min": min_from_seconds(intervention["max_time_loss_sec"]),
            "congestion_index": congestion_index(intervention),
            "queue_impact": "Medium",
            "service_reliability": "Improved",
            "validation_interpretation": f"Intervention OSM visual scenario applies a Decision Engine response condition and produces lower average duration and time-loss indicators than disruption, with about {improvement_pct:.1f} percent lower average SUMO time-loss. Because completed trip counts differ across scenarios, this is a scenario-level comparison rather than a like-for-like traffic-volume experiment. Output was generated from finalized command-line SUMO XML. SUMO outputs are scenario-estimated impacts and do not prove real-world operational success.",
        },
    ])

    sumo_kpis = pd.DataFrame([
        {"metric": "Average Delay", "description": f"Average SUMO time-loss converted to minutes for completed trips in the finalized command-line Route 128 OSM visual scenario, contextualized with final model-baseline evidence from {evidence_period}.", "baseline_target": "Lower than disruption", "intervention_goal": "Reduce compared with disruption"},
        {"metric": "Maximum Delay", "description": f"Maximum SUMO time-loss converted to minutes for completed trips in the finalized command-line Route 128 OSM visual scenario, contextualized with final model-baseline evidence from {evidence_period}.", "baseline_target": "Lower than disruption", "intervention_goal": "Reduce peak time-loss compared with disruption"},
        {"metric": "Congestion Index", "description": "Scenario-level congestion proxy derived from average trip duration relative to the baseline scenario using finalized command-line SUMO outputs.", "baseline_target": "Baseline reference around 35", "intervention_goal": "Reduce compared with disruption"},
        {"metric": "Queue Impact", "description": "Qualitative interpretation of scenario traffic pressure, completed trips, and halting behaviour.", "baseline_target": "Low", "intervention_goal": "Reduce queue or pressure impact compared with disruption"},
        {"metric": "Service Reliability", "description": "Qualitative interpretation of whether the scenario appears stable, degraded, or improved based on SUMO summary and tripinfo outputs.", "baseline_target": "Stable", "intervention_goal": "Improve compared with disruption"},
        {"metric": "Intervention Effectiveness", "description": "Estimated scenario improvement calculated as (disruption average time-loss - intervention average time-loss) / disruption average time-loss * 100.", "baseline_target": "N/A", "intervention_goal": "Show positive scenario-level improvement without claiming real-world operational proof"},
    ])

    sumo_scenarios.to_csv(DATA_PROCESSED / "sumo_scenarios.csv", index=False)
    sumo_validation_results.to_csv(DATA_PROCESSED / "sumo_validation_results.csv", index=False)
    sumo_kpis.to_csv(DATA_PROCESSED / "sumo_kpis.csv", index=False)

    print("Wrote SUMO dashboard-ready CSV outputs.")
else:
    print("WRITE_OUTPUTS is False. Review the metrics above before enabling CSV output writing.")


Wrote SUMO dashboard-ready CSV outputs.


## 9. Academic Interpretation

Use the following wording in the report and dashboard handoff:

> The SUMO prototype is an evidence-informed scenario validation layer. It uses project outputs to select a suitable bus route and connect delay evidence to Decision Engine recommendations, but the SUMO traffic flows, bus headways, and intervention settings are manually configured scenario assumptions rather than calibrated real-world Route 128 operations.

> The SUMO comparison evaluates relative scenario behaviour under different operational assumptions. The disruption scenario intentionally increases traffic pressure, while the intervention scenario represents a managed response condition. The lower average duration and time-loss values in the intervention scenario indicate improved scenario-level performance, but the difference in completed trip counts means the result should be interpreted as an estimated scenario impact rather than a like-for-like traffic-volume experiment.

> SUMO outputs are scenario-estimated impacts and do not prove real-world operational success.


## 10. RQ2 Alignment

RQ2 asks how predicted public transport delay risk can be translated into actionable operational decision support for Auckland transport services.

The SUMO layer supports RQ2 by demonstrating the final validation step:

```text
Delay evidence / predicted risk
-> Decision Engine recommendation
-> baseline, disruption, and intervention SUMO scenarios
-> scenario-estimated impact shown in the dashboard
```

SUMO does not prove the recommendation will succeed in real operations. It shows that the prototype can translate a recommendation into a controlled scenario comparison and communicate the estimated impact.
